# VisionGuard — Generate chest-X-ray reports with Qwen2-VL-7B

Runs the Qwen2-VL-7B-Instruct vision-language model on the first 50 frontal images of the public IU Chest X-Ray dataset and writes a JSON mapping `{sha256_of_image: generated_report_text}`.

**Setup inside this notebook (right-hand panel):**
1. Accelerator → **GPU T4 x1**
2. Internet → On
3. Add Input → search *indiana university chest* → attach `raddar/chest-xrays-indiana-university`

Then **Run All**. Output lands at `/kaggle/working/vlm_generated_reports.json`.

Not clinically validated. Research prototype only.

In [ ]:
!pip -q install --upgrade 'transformers>=4.45' accelerate 'qwen-vl-utils' bitsandbytes 2>&1 | tail -3

In [ ]:
import hashlib, json, torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

candidates = [p for p in Path('/kaggle/input').rglob('images_normalized') if p.is_dir()]
assert candidates, f'No images_normalized folder under /kaggle/input. Contents: {list(Path("/kaggle/input").rglob("*"))[:30]}'
DATASET_IN = candidates[0]
OUT_PATH = Path('/kaggle/working/vlm_generated_reports.json')
MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'   # 7B OOMs a free T4; 2B fits with headroom
N_IMAGES = 50

print('Dataset mount:', DATASET_IN)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
print('Loading Qwen2-VL-7B — first run downloads ~15 GB (bf16) ...')
# Cap vision tower input so a single high-res chest X-ray does not blow 15 GB VRAM.
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=896*896)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
model.eval()
print('Model ready.')

In [ ]:
PROMPT = (
    'You are a radiology assistant. Describe this chest X-ray strictly in the '
    'following format and nothing else:\n'
    'FINDINGS: <2-4 short sentences about observable findings>\n'
    'IMPRESSION: <1-2 short sentences summarising the clinical impression>\n'
    'If a finding is not visible, state that it is absent. Do not invent lab '
    'values, prior studies, or patient history. This is a research prototype, '
    'not for clinical use.'
)

import gc
images = sorted(DATASET_IN.glob('*.png'))[:N_IMAGES]
print(f'Processing {len(images)} images')

results = {}
for i, img_path in enumerate(images, 1):
    raw = img_path.read_bytes()
    key = hashlib.sha256(raw).hexdigest()
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(img_path)},
            {'type': 'text', 'text': PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    reply = processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    results[key] = {'image': img_path.name, 'report': reply}
    print(f'[{i:>2}/{len(images)}] {img_path.name}  ->  {reply[:100]}...')
    del inputs, out
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
import base64, json
from IPython.display import HTML, FileLink, display

OUT_PATH.write_text(json.dumps(results, indent=2))
print(f'Wrote {len(results)} entries to {OUT_PATH}')

b64 = base64.b64encode(json.dumps(results, indent=2).encode('utf-8')).decode('ascii')
display(HTML(f'''
<a id="auto_dl" download="vlm_generated_reports.json"
   href="data:application/json;base64,{b64}">Click here if download does not start.</a>
<script>setTimeout(() => document.getElementById("auto_dl").click(), 500);</script>
'''))
display(FileLink(str(OUT_PATH)))

print('\nSave the downloaded file to:')
print('  datasets/iu_xray/vlm_generated_reports.json  (inside the local repo)')